# Grounded Carbon-Market QA: data + train + evaluate (Kaggle)
**Settings → Accelerator: GPU T4 x2, Internet: ON.** Add-ons → Secrets: `HF_TOKEN` (optional: `GROQ_API_KEY`).
Attach the uploaded `carbon-grounded` dataset, then **Run All** (about 3 h).

All LLM work (QA generation, hallucination gate, unanswerable verification, evaluation judge) runs on a
**local Qwen2.5-14B served by Ollama on the Kaggle GPU**, so no API limits. The question–answer pairs already
generated with Groq gpt-oss-120b on the laptop are reused from `data/work/llm_cache.jsonl`.

In [ ]:
import glob, os, shutil
src = os.path.dirname(glob.glob('/kaggle/input/**/config.py', recursive=True)[0])
shutil.copytree(src, '/kaggle/working/proj', dirs_exist_ok=True)
os.chdir('/kaggle/working/proj')
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
for k in ('HF_TOKEN', 'GROQ_API_KEY'):
    try: os.environ[k] = s.get_secret(k)
    except Exception: print('secret not set (ok if optional):', k)
LOCAL = 'ollama_chat/qwen2.5:14b'
os.environ.update({
    'LITELLM_LOCAL_MODEL_COST_MAP': 'True',
    'GEN_MODEL': LOCAL, 'GATE_MODEL': LOCAL, 'VERIFY_MODEL': LOCAL, 'JUDGE_MODEL': LOCAL,
    'AUDIT_JUDGE': 'litellm:' + LOCAL,
    'CACHE_FALLBACK_MODELS': 'groq/openai/gpt-oss-120b',   # reuse laptop answers
    'LLM_CONCURRENCY': '4', 'LLM_TIMEOUT': '600', 'OLLAMA_NUM_PARALLEL': '4',
})
print(sorted(os.listdir('.')), os.path.exists('data/work/chunks.jsonl'))

In [ ]:
!pip install -q aligntune unsloth rank_bm25 sentence-transformers litellm python-dotenv auditkit "curatorkit[all]" 2>&1 | tail -3
!python -c "import aligntune, unsloth, torch; print('ok', torch.cuda.get_device_name(0))"

In [ ]:
# Local LLM server (Ollama) on the GPU
!apt-get -qq install -y zstd > /dev/null; curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import subprocess, time, requests
def start_ollama():
    p = subprocess.Popen(['ollama', 'serve'], stdout=open('ollama.log', 'w'), stderr=subprocess.STDOUT)
    for _ in range(60):
        try: requests.get('http://localhost:11434'); return p
        except Exception: time.sleep(1)
ollama = start_ollama()
!ollama pull qwen2.5:14b 2>&1 | tail -1
!python -c "from common.llm import chat; import os; print(chat(os.environ['GEN_MODEL'], [{'role':'user','content':'Say OK'}], max_tokens=10))"

In [ ]:
# Data steps 2-4 (about 45-60 min)
!python scripts/02_curate.py --engine fallback
!python scripts/03_unanswerable.py
!python scripts/04_build_sets.py

In [ ]:
# Free the GPU for training
ollama.terminate(); ollama.wait(); time.sleep(5)
!pkill -f ollama; nvidia-smi --query-gpu=memory.used --format=csv

In [ ]:
# 5-minute smoke test: proves AlignTune + Unsloth + data format work before the long run
!python scripts/05_train.py --max-samples 40 --epochs 1

In [ ]:
# Full training (~1 h on T4). If Unsloth fails above, add: --backend trl
!python scripts/05_train.py

In [ ]:
# 3 systems on the test set
!python scripts/06_generate.py

In [ ]:
# Scoring with the local judge (restart Ollama after generation freed the GPU)
import gc, torch; gc.collect(); torch.cuda.empty_cache()
ollama = start_ollama()
!python scripts/07_evaluate.py
from IPython.display import Image, Markdown, display
display(Markdown(open('outputs/results/report.md').read())); display(Image('outputs/results/chart.png'))

In [ ]:
# Save everything (download results.zip from the Output tab)
!zip -qr /kaggle/working/results.zip outputs/sft outputs/test_candidates.csv outputs/preds outputs/results outputs/adapter data/work/funnel.json data/work/unanswerable_stats.json data/work/qa_raw.jsonl data/work/unanswerable.jsonl
!ls -la /kaggle/working/results.zip

In [ ]:
# Demo with a public link (stop the cell to end it)
ollama.terminate()
!pip install -q gradio
!python app.py